# Assignment 1: Data Parsing, Cleansing and Integration
## Tasks 1 and 2
#### Student Name: Prakash Sudarsan raj
#### Student ID: s4135999


Environment: Python 3 and Jupyter notebook

Libraries used: please include the main libraries you used in your assignment, e.g.,:
* pandas
* re
* numpy

## Introduction
This assignment focuses on parsing and cleaning an XML dataset using Python. The dataset contains clothing reviews with multiple issues such as incorrect data types, missing values, and spelling mistakes.

During the process, I found several problems including:
- Numeric values stored as text
- Missing values in some columns
- Invalid values in ratings and costs
- Typographical errors in category and section names

To fix these issues, I used data type conversion, value correction, and mapping techniques. An error log was also created to track all changes made to the dataset.

## Importing libraries 

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import xml.etree.ElementTree as ET
from pathlib import Path

In [2]:
base_path = Path(".")
xml_path = base_path / "S4135999_dataset1.xml"

## Task 1. Parsing Data

### 1.1. Examining and loading data
The XML file has a root element called "Clothes" and contains multiple "Response" elements. Each "Response" represents one record in the dataset.

Each record includes fields such as ClothID, Age, Review Title, Customer Rating, Category, Cost, and Recommended IND. Some values like Section and Department are stored inside a nested "Store" tag.

This structure needs to be parsed carefully to extract all values correctly into a tabular format.

In [3]:
tree = ET.parse(xml_path)
root = tree.getroot()

print(root.tag)
print(len(root.findall(".//Response")))

Clothes
6435


In [4]:
first_response = root.find(".//Response")
for child in first_response:
    print(child.tag, child.text)

Store None
ClothID 892.0
Age 39
Review_Title So soft and figure-flattering
Customer_Rating 5.0
Recommended_IND 1
Positive_Review_Count 0.0
Category Fine gauge
Online_Time 10.76120366462436
Cost 38.46810848421382


In [5]:
rows = []
df = pd.DataFrame(rows)
df.head()

""


In [6]:
rows = []

for response in root.findall(".//Response"):
    rows.append({
        "ClothID": response.findtext("ClothID"),
        "Age": response.findtext("Age"),
        "Review Title": response.findtext("Review_Title"),
        "Customer Rating": response.findtext("Customer_Rating"),
        "Positive Review Count": response.findtext("Positive_Review_Count"),
        "Section": response.findtext("Store/Section"),
        "Department": response.findtext("Store/Department"),
        "Category": response.findtext("Category"),
        "Online Time": response.findtext("Online_Time"),
        "Cost": response.findtext("Cost"),
        "Recommended IND": response.findtext("Recommended_IND")
    })

df = pd.DataFrame(rows)
df.head()

,ClothID,Age,Review Title,Customer Rating,Positive Review Count,Section,Department,Category,Online Time,Cost,Recommended IND
0,892.0,39,So soft and figure-flattering,5.0,0.0,General,Tops,Fine gauge,10.76120366462436,38.46810848421382,1
1,867.0,28,"Great shirt, but don't expect it to last",3.0,0.0,General Petite,Tops,Knits,3.5937125828672483,47.80342951552357,1
2,997.0,49,Beautiful!,5.0,0.0,General Petite,Bottoms,Skirts,5.445432129450735,52.32264129906318,1
3,1077.0,26,So much better in person,5.0,5.0,General,Dresses,Dresses,4.960395048567458,73.03712399613934,1
4,1033.0,35,Nope!,2.0,0.0,General,Bottoms,Jeans,21.308399722048907,51.940232446848746,0


After parsing the XML file, the data was converted into a pandas DataFrame.

The dataset contains 6435 rows and multiple columns representing different attributes of each review. Each row corresponds to a single customer review.

In [7]:
print(df.shape)
print(df.columns.tolist())
print(df.isna().sum())
df.info()

(6435, 11)
['ClothID', 'Age', 'Review Title', 'Customer Rating', 'Positive Review Count', 'Section', 'Department', 'Category', 'Online Time', 'Cost', 'Recommended IND']
ClothID                  0
Age                      0
Review Title             0
Customer Rating          0
Positive Review Count    0
Section                  0
Department               0
Category                 0
Online Time              0
Cost                     0
Recommended IND          0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6435 entries, 0 to 6434
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   ClothID                6435 non-null   object
 1   Age                    6435 non-null   object
 2   Review Title           6435 non-null   object
 3   Customer Rating        6435 non-null   object
 4   Positive Review Count  6435 non-null   object
 5   Section                6435 non-null   object
 6   D

### 1.2 Parsing data into the required format

In this step, the data types of relevant columns were converted into numeric format.

Since XML stores all values as text, conversion is required for proper analysis. Invalid values were handled using errors="coerce", which converts them into missing values (NaN).

In [8]:
numeric_cols = [
    "ClothID", "Age", "Customer Rating", "Positive Review Count",
    "Online Time", "Cost", "Recommended IND"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [9]:
print(df.columns.tolist())

['ClothID', 'Age', 'Review Title', 'Customer Rating', 'Positive Review Count', 'Section', 'Department', 'Category', 'Online Time', 'Cost', 'Recommended IND']


......

## Task 2. Auditing and cleansing the loaded data
In this section, the dataset is checked for errors and cleaned accordingly.

An error log is created to record all the issues found and the fixes applied. The main problems handled include missing values, incorrect data types, invalid values, and spelling mistakes in categorical columns.


In [10]:
error_log = []

def log_error(index, cloth_id, column, original, modified, error_type, fixing):
    error_log.append({
        "datasetNo": "dataset1",
        "indexOfdf": index,
        "Id": cloth_id,
        "ColumnName": column,
        "Original": original,
        "Modified": modified,
        "ErrorType": error_type,
        "Fixing": fixing
    })

### Fixing Positive Review Count

Some values were not integers or were missing. Non-integer values were rounded, and missing values were replaced with 0.

In [11]:
invalid_idx = df[df["Positive Review Count"] % 1 != 0].index

for idx in invalid_idx:
    old = df.at[idx, "Positive Review Count"]
    new = int(round(old))

    log_error(
        idx,
        df.at[idx, "ClothID"],
        "Positive Review Count",
        old,
        new,
        "Invalid Format",
        "Rounded non-integer review count to nearest integer."
    )

    df.at[idx, "Positive Review Count"] = new


nan_idx = df[df["Positive Review Count"].isna()].index

for idx in nan_idx:
    old = df.at[idx, "Positive Review Count"]
    new = 0

    log_error(
        idx,
        df.at[idx, "ClothID"],
        "Positive Review Count",
        old,
        new,
        "Missing Value",
        "Replaced missing review count with 0."
    )

    df.at[idx, "Positive Review Count"] = new

df["Positive Review Count"] = df["Positive Review Count"].astype("Int64")

### Fixing Customer Rating

Customer ratings should be integers between 1 and 5. Invalid values were corrected by rounding and limiting them within this range.

In [12]:
invalid_rating_idx = df[~df["Customer Rating"].isin([1, 2, 3, 4, 5])].index

for idx in invalid_rating_idx:
    old = df.at[idx, "Customer Rating"]

    if pd.notna(old):
        new = int(round(old))
        new = max(1, min(5, new))
    else:
        new = 3

    log_error(
        idx,
        df.at[idx, "ClothID"],
        "Customer Rating",
        old,
        new,
        "Invalid Format",
        "Rounded rating to nearest valid integer between 1 and 5."
    )

    df.at[idx, "Customer Rating"] = new

df["Customer Rating"] = df["Customer Rating"].astype("Int64")

### Fixing Cost

Some cost values were invalid (less than or equal to 0). These were replaced using the median cost of the same category to maintain consistency.

In [13]:
invalid_cost_idx = df[df["Cost"] <= 0].index

for idx in invalid_cost_idx:
    old = df.at[idx, "Cost"]
    category = df.at[idx, "Category"]

    category_median = df[(df["Category"] == category) & (df["Cost"] > 0)]["Cost"].median()

    if pd.isna(category_median):
        new = df[df["Cost"] > 0]["Cost"].median()
    else:
        new = category_median

    log_error(
        idx,
        df.at[idx, "ClothID"],
        "Cost",
        old,
        new,
        "Incorrect Value",
        "Replaced invalid cost using median cost of the same category."
    )

    df.at[idx, "Cost"] = new

df["Cost"] = df["Cost"].astype(float)

### Fixing Section and Category Errors

Some values had spelling mistakes such as "Initmates" and "Blousse". These were corrected using predefined mappings to ensure consistency.

In [14]:
section_map = {
    "Initmates": "Intimates"
}

for wrong, correct in section_map.items():
    idxs = df[df["Section"] == wrong].index

    for idx in idxs:
        old = df.at[idx, "Section"]

        log_error(
            idx,
            df.at[idx, "ClothID"],
            "Section",
            old,
            correct,
            "Misspelling",
            "Corrected inconsistent section label."
        )

        df.at[idx, "Section"] = correct



category_map = {
    "Blousse": "Blouses",
    "Swmi": "Swim"
}

for wrong, correct in category_map.items():
    idxs = df[df["Category"] == wrong].index

    for idx in idxs:
        old = df.at[idx, "Category"]

        log_error(
            idx,
            df.at[idx, "ClothID"],
            "Category",
            old,
            correct,
            "Misspelling",
            "Corrected inconsistent category label."
        )

        df.at[idx, "Category"] = correct

### Fixing Remaining Category Errors

Additional typos like "kSirts", "Dersses", and "Kints" were corrected to their proper values.

In [15]:
more_category_map = {
    "kSirts": "Skirts",
    "Dersses": "Dresses",
    "Kints": "Knits",
    "Skrits": "Skirts"
}

for wrong, correct in more_category_map.items():
    idxs = df[df["Category"] == wrong].index

    for idx in idxs:
        old = df.at[idx, "Category"]

        log_error(
            idx,
            df.at[idx, "ClothID"],
            "Category",
            old,
            correct,
            "Misspelling",
            "Corrected inconsistent category label."
        )

        df.at[idx, "Category"] = correct

In [16]:
df[df["Category"] == "Chemises"][["ClothID", "Review Title", "Department", "Category"]]

,ClothID,Review Title,Department,Category
916,10.0,Comfy and cute,Intimate,Chemises


### Fixing Department Inconsistency

The value "Intimate" was corrected to "Intimates" to ensure consistent naming across the dataset.

In [17]:
dept_map = {
    "Intimate": "Intimates"
}

for wrong, correct in dept_map.items():
    idxs = df[df["Department"] == wrong].index

    for idx in idxs:
        old = df.at[idx, "Department"]

        log_error(
            idx,
            df.at[idx, "ClothID"],
            "Department",
            old,
            correct,
            "Inconsistent Label",
            "Standardised department naming."
        )

        df.at[idx, "Department"] = correct

### Data Validation

After cleaning, the dataset was checked for:
- Missing values
- Duplicate records
- Valid ranges of ratings and recommendations
- Consistency in categorical values

In [18]:
print("Missing values:\n", df.isna().sum())
print("\nExact duplicate rows:", df.duplicated().sum())
print("\nDepartment values:\n", df["Department"].value_counts())
print("\nCategory tail:\n", df["Category"].value_counts().tail(15))
print("\nRating values:\n", df["Customer Rating"].value_counts().sort_index())
print("\nRecommended IND values:\n", df["Recommended IND"].value_counts())
print("\nData types:\n", df.dtypes)

Missing values:
 ClothID                  0
Age                      0
Review Title             0
Customer Rating          0
Positive Review Count    0
Section                  0
Department               0
Category                 0
Online Time              0
Cost                     0
Recommended IND          0
dtype: int64

Exact duplicate rows: 0

Department values:
 Tops         2860
Dresses      1736
Bottoms      1054
Intimates     439
Jackets       314
Trend          32
Name: Department, dtype: int64

Category tail:
 Pants         364
Jeans         331
Fine gauge    289
Skirts        285
Jackets       216
Lounge        170
Swim           99
Outerwear      98
Shorts         74
Sleep          57
Layering       39
Legwear        39
Intimates      34
Trend          32
Chemises        1
Name: Category, dtype: int64

Rating values:
 1     207
2     442
3     804
4    1430
5    3552
Name: Customer Rating, dtype: Int64

Recommended IND values:
 1    5289
0    1146
Name: Recommended IND, 

In [19]:
error_df = pd.DataFrame(error_log)

required_cols = [
    "datasetNo", "indexOfdf", "Id", "ColumnName",
    "Original", "Modified", "ErrorType", "Fixing"
]

error_df = error_df[required_cols]

df.to_csv("S4135999_dataset1_solution.csv", index=False)
error_df.to_csv("S4135999_errorlist.csv", index=False)

error_df.head()

,datasetNo,indexOfdf,Id,ColumnName,Original,Modified,ErrorType,Fixing
0,dataset1,1374,877.0,Positive Review Count,2.5,2,Invalid Format,Rounded non-integer review count to nearest in...
1,dataset1,2221,1066.0,Positive Review Count,1.5,2,Invalid Format,Rounded non-integer review count to nearest in...
2,dataset1,3381,1100.0,Positive Review Count,0.5,0,Invalid Format,Rounded non-integer review count to nearest in...
3,dataset1,3430,883.0,Positive Review Count,0.5,0,Invalid Format,Rounded non-integer review count to nearest in...
4,dataset1,4125,936.0,Positive Review Count,0.5,0,Invalid Format,Rounded non-integer review count to nearest in...


In [20]:
print("Exact duplicate rows:", df.duplicated().sum())

Exact duplicate rows: 0


In [21]:
invalid_rating = df[~df["Customer Rating"].isin([1, 2, 3, 4, 5])]
invalid_recommend = df[~df["Recommended IND"].isin([0, 1])]
invalid_cost = df[df["Cost"] <= 0]

In [22]:
df.to_csv("S4135999_dataset1_solution.csv", index=False)
error_df.to_csv("S4135999_errorlist.csv", index=False)

In [23]:
print(df["Category"].value_counts().tail(15))
print(df["Department"].value_counts())
print("Exact duplicate rows:", df.duplicated().sum())

Pants         364
Jeans         331
Fine gauge    289
Skirts        285
Jackets       216
Lounge        170
Swim           99
Outerwear      98
Shorts         74
Sleep          57
Layering       39
Legwear        39
Intimates      34
Trend          32
Chemises        1
Name: Category, dtype: int64
Tops         2860
Dresses      1736
Bottoms      1054
Intimates     439
Jackets       314
Trend          32
Name: Department, dtype: int64
Exact duplicate rows: 0


In [24]:
df[df["Category"] == "Chemises"]

,ClothID,Age,Review Title,Customer Rating,Positive Review Count,Section,Department,Category,Online Time,Cost,Recommended IND
916,10.0,38,Comfy and cute,4,0,Intimates,Intimates,Chemises,6.00113,50.636088,1


In [25]:
print("Missing values:\n", df.isna().sum())
print("\nCategory unique:", df["Category"].nunique())
print("\nDepartment unique:", df["Department"].nunique())
print("\nRating values:\n", df["Customer Rating"].value_counts().sort_index())
print("\nRecommended IND:\n", df["Recommended IND"].value_counts())

Missing values:
 ClothID                  0
Age                      0
Review Title             0
Customer Rating          0
Positive Review Count    0
Section                  0
Department               0
Category                 0
Online Time              0
Cost                     0
Recommended IND          0
dtype: int64

Category unique: 19

Department unique: 6

Rating values:
 1     207
2     442
3     804
4    1430
5    3552
Name: Customer Rating, dtype: Int64

Recommended IND:
 1    5289
0    1146
Name: Recommended IND, dtype: int64


In [26]:
print(df.shape)
print(df.columns.tolist())

(6435, 11)
['ClothID', 'Age', 'Review Title', 'Customer Rating', 'Positive Review Count', 'Section', 'Department', 'Category', 'Online Time', 'Cost', 'Recommended IND']


## Saving data
The cleaned dataset and the error log were saved as CSV files for submission.

In [27]:
df.to_csv("S4135999_dataset1_solution.csv", index=False)

error_df = pd.DataFrame(error_log)
error_df.to_csv("S4135999_errorlist.csv", index=False)

## Summary
Give a short summary and anything you would like to talk about the assessment here.